In [85]:
import pandas as pd
import numpy as np

In [86]:
df = pd.read_csv('sentiment_data.csv', sep=',', encoding='utf-8')

In [87]:
df.head()

,Unnamed: 0,Comment,Sentiment
0,0,lets forget apple pay required brand new iphon...,1
1,1,nz retailers don’t even contactless credit car...,0
2,2,forever acknowledge channel help lessons ideas...,2
3,3,whenever go place doesn’t take apple pay doesn...,0
4,4,apple pay convenient secure easy use used kore...,2


In [88]:
df.drop(['Unnamed: 0'], axis=1, inplace=True)

In [89]:
df.head()

,Comment,Sentiment
0,lets forget apple pay required brand new iphon...,1
1,nz retailers don’t even contactless credit car...,0
2,forever acknowledge channel help lessons ideas...,2
3,whenever go place doesn’t take apple pay doesn...,0
4,apple pay convenient secure easy use used kore...,2


In [90]:
df.isna().sum()

Comment      217
Sentiment      0
dtype: int64

In [91]:
df[df['Comment'].isna()]

,Comment,Sentiment
1014,NaN,1
4732,NaN,1
7414,NaN,2
7431,NaN,1
7496,NaN,1
...,...,...
223441,NaN,1
228310,NaN,2
233352,NaN,1
235494,NaN,1


In [92]:
df.shape

(241145, 2)

In [93]:
df.describe()

,Sentiment
count,241145.000000
mean,1.198822
std,0.785110
min,0.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,2.000000


In [94]:
df.dropna(inplace=True)

In [95]:
df.shape

(240928, 2)

In [96]:
df.isna().sum()

Comment      0
Sentiment    0
dtype: int64

In [97]:
# !pip install nltk

In [98]:
df['Comment'] = df['Comment'].apply(lambda x: x.lower())

In [99]:
df['Comment'].head()

0    lets forget apple pay required brand new iphon...
1    nz retailers don’t even contactless credit car...
2    forever acknowledge channel help lessons ideas...
3    whenever go place doesn’t take apple pay doesn...
4    apple pay convenient secure easy use used kore...
Name: Comment, dtype: str

In [100]:
import re

has_html_tag = df['Comment'].str.contains(r'<.*?>', regex=True)
print(f"Number of comments with HTML tags: {has_html_tag.sum()}")

Number of comments with HTML tags: 0


In [101]:

has_url = df['Comment'].str.contains(r'https?://[^\s]+', regex=True)
print(f"Number of comments with URLs: {has_url.sum()}")

Number of comments with URLs: 0


In [102]:
import string

has_punctuation = df['Comment'].str.contains(f"[{re.escape(string.punctuation)}]")
print(f"Number of comments with punctuation: {has_punctuation.sum()}")

Number of comments with punctuation: 0


In [103]:
chat_words = {
    "afaik": "as far as i know", "afk": "away from keyboard",
    "asap": "as soon as possible", "atm": "at the moment",
    "b4": "before", "bc": "because", "bday": "birthday",
    "brb": "be right back", "btw": "by the way", "bff": "best friends forever",
    "cya": "see you", "cu": "see you", "diy": "do it yourself",
    "dm": "direct message", "eta": "estimated time of arrival",
    "fomo": "fear of missing out", "ftw": "for the win",
    "fyi": "for your information", "gg": "good game",
    "gtg": "got to go", "g2g": "got to go", "gr8": "great",
    "hbd": "happy birthday", "hmu": "hit me up",
    "idc": "i do not care", "idk": "i do not know",
    "ikr": "i know right", "ily": "i love you",
    "imo": "in my opinion", "imho": "in my humble opinion",
    "irl": "in real life", "jk": "just kidding",
    "lmk": "let me know", "lol": "laughing out loud",
    "luv": "love", "l8r": "later", "msg": "message",
    "nvm": "never mind", "np": "no problem",
    "omg": "oh my god", "omw": "on my way",
    "ppl": "people", "pls": "please", "plz": "please",
    "rn": "right now", "rofl": "rolling on the floor laughing",
    "smh": "shaking my head", "tbh": "to be honest",
    "thx": "thanks", "ty": "thank you", "tysm": "thank you so much",
    "ttyl": "talk to you later", "tmi": "too much information",
    "u": "you", "ur": "your", "r": "are",
    "w8": "wait", "wyd": "what are you doing",
    "wbu": "what about you", "wth": "what the hell",
    "wtf": "what the fuck", "yolo": "you only live once",
    "2day": "today", "2moro": "tomorrow", "2nite": "tonight",
    "4u": "for you", "b/c": "because", "cuz": "because",
    "gonna": "going to", "wanna": "want to", "gotta": "got to",
    "kinda": "kind of", "sorta": "sort of", "lemme": "let me",
    "gimme": "give me", "dunno": "do not know",
    "aight": "alright", "k": "okay", "kk": "okay",
    "np": "no problem", "nm": "not much", "wassup": "what is up",
    "sup": "what is up", "howdy": "hello",
    "xoxo": "hugs and kisses", "yw": "you are welcome",
    "bday": "birthday", "congrats": "congratulations",
    "def": "definitely", "obv": "obviously", "obvi": "obviously",
    "prob": "probably", "probs": "probably", "totes": "totally",
    "srsly": "seriously", "rly": "really", "vry": "very",
    "abt": "about", "acct": "account", "addy": "address",
    "app": "application", "bbl": "be back later",
    "bday": "birthday", "convo": "conversation",
    "fam": "family", "fav": "favorite", "fave": "favorite",
    "grats": "congratulations", "hru": "how are you",
    "hw": "homework", "ic": "i see", "iirc": "if i recall correctly",
    "jic": "just in case", "jsyk": "just so you know",
    "msg": "message", "nbd": "no big deal", "nsfw": "not safe for work",
    "ofc": "of course", "outta": "out of", "peeps": "people",
    "pic": "picture", "pics": "pictures", "pov": "point of view",
    "qt": "cutie", "sec": "second", "sis": "sister", "bro": "brother",
    "tho": "though", "thru": "through", "til": "until",
    "w/": "with", "w/o": "without", "y": "why",
    "ya": "you", "yea": "yeah", "yep": "yes", "nope": "no",
    "cmon": "come on", "dont": "do not", "cant": "cannot",
    "wont": "will not", "im": "i am", "ive": "i have",
    "id": "i would", "ill": "i will"
}

pattern = r'\b(' + '|'.join(re.escape(k) for k in chat_words.keys()) + r')\b'
has_chat_words = df['Comment'].str.contains(pattern, regex=True)
print(f"Number of comments with chat words: {has_chat_words.sum()}")

/var/folders/hg/ybvpz2sd5pxg82v04p1v28w00000gn/T/ipykernel_53793/1325410124.py:62: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_chat_words = df['Comment'].str.contains(pattern, regex=True)


Number of comments with chat words: 43899


In [104]:
pattern = r'\b(' + '|'.join(re.escape(k) for k in chat_words.keys()) + r')\b'
matched_chat_words = df['Comment'].str.findall(pattern, flags=re.IGNORECASE)
matched_chat_words.head()

0    []
1    []
2    []
3    []
4    []
Name: Comment, dtype: object

In [105]:
mask = df['Comment'].str.contains('ive', case=False, na=False)
df[mask]

,Comment,Sentiment
0,lets forget apple pay required brand new iphon...,1
9,cambodia universal qr code system scan send mo...,1
11,lab exciting thing ive seen reallly going shak...,2
13,used time linus smartest guy room video clearl...,2
25,dan man saving day riley needs give theme like...,2
...,...,...
241130,people say modi looking defensive days,1
241134,objective cover everything akhlaq ramalingam q...,2
241137,engine growth modi unveils indias first electr...,2
241138,modi promised lok sabha elections best orop gi...,2


In [106]:
matched_chat_words = matched_chat_words[matched_chat_words.str.len() > 0]

In [107]:
matched_chat_words

11          [ive]
12           [im]
19        [gonna]
29           [id]
40         [dont]
           ...   
241113     [wont]
241122     [dont]
241133     [dont]
241134     [dont]
241135     [wont]
Name: Comment, Length: 43899, dtype: object

In [108]:
import re

pattern = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in chat_words.keys()) + r')\b',
    flags=re.IGNORECASE
)

def chat_word_conversion(txt):
    return pattern.sub(lambda m: chat_words[m.group(0).lower()], txt)


In [109]:
df['cleaned_comment'] = df['Comment'].apply(chat_word_conversion)

In [110]:
df.head()

,Comment,Sentiment,cleaned_comment
0,lets forget apple pay required brand new iphon...,1,lets forget apple pay required brand new iphon...
1,nz retailers don’t even contactless credit car...,0,nz retailers don’t even contactless credit car...
2,forever acknowledge channel help lessons ideas...,2,forever acknowledge channel help lessons ideas...
3,whenever go place doesn’t take apple pay doesn...,0,whenever go place doesn’t take apple pay doesn...
4,apple pay convenient secure easy use used kore...,2,apple pay convenient secure easy use used kore...


In [111]:
pattern = r'\b(' + '|'.join(re.escape(k) for k in chat_words.keys()) + r')\b'
has_chat_words = df['cleaned_comment'].str.contains(pattern, regex=True)
print(f"Number of comments with chat words: {has_chat_words.sum()}")

/var/folders/hg/ybvpz2sd5pxg82v04p1v28w00000gn/T/ipykernel_53793/337416178.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_chat_words = df['cleaned_comment'].str.contains(pattern, regex=True)


Number of comments with chat words: 0


In [112]:
pattern = r'\b(' + '|'.join(re.escape(k) for k in chat_words.keys()) + r')\b'
matched_chat_words = df['cleaned_comment'].str.findall(pattern, flags=re.IGNORECASE)
matched_chat_words.head()

0    []
1    []
2    []
3    []
4    []
Name: cleaned_comment, dtype: object

In [113]:
matched_chat_words = matched_chat_words[matched_chat_words.str.len() > 0]

In [114]:
matched_chat_words

Series([], Name: cleaned_comment, dtype: object)

In [115]:
df.head()

,Comment,Sentiment,cleaned_comment
0,lets forget apple pay required brand new iphon...,1,lets forget apple pay required brand new iphon...
1,nz retailers don’t even contactless credit car...,0,nz retailers don’t even contactless credit car...
2,forever acknowledge channel help lessons ideas...,2,forever acknowledge channel help lessons ideas...
3,whenever go place doesn’t take apple pay doesn...,0,whenever go place doesn’t take apple pay doesn...
4,apple pay convenient secure easy use used kore...,2,apple pay convenient secure easy use used kore...


In [118]:
df[df['cleaned_comment'].str.contains(r'\bomg\b', case=False, regex=True, na=False)]

,Comment,Sentiment,cleaned_comment


In [119]:
df['Comment'] = df['cleaned_comment']
df = df.drop(columns=['cleaned_comment'])

In [120]:
df.head()

,Comment,Sentiment
0,lets forget apple pay required brand new iphon...,1
1,nz retailers don’t even contactless credit car...,0
2,forever acknowledge channel help lessons ideas...,2
3,whenever go place doesn’t take apple pay doesn...,0
4,apple pay convenient secure easy use used kore...,2


In [124]:
!pip install pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 3.3 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [125]:
from spellchecker import SpellChecker

spell = SpellChecker()

def correct_spelling(text):
    words = text.split()
    corrected_words = [spell.correction(word) for word in words]
    return ' '.join(corrected_words)

In [127]:
def check_spelling(text):
    words = text.split()
    corrected_words = [{ 'word': word, 'corrected': spell.correction(word) } for word in words]

In [128]:
result = df['Comment'].apply(check_spelling)

KeyboardInterrupt: 

In [129]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

[nltk_data] Downloading package stopwords to /Users/nayan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [130]:
df['Comment'] = df['Comment'].apply(remove_stopwords)

In [ ]:
# !pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 4.4 MB/s eta 0:00:00-:--:--

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [133]:
import emoji

def has_emoji(text):
    return bool(emoji.emoji_list(text))

In [134]:
df['has_emoji'] = df['Comment'].apply(has_emoji)
print(f"Number of comments with emojis: {df['has_emoji'].sum()}")

Number of comments with emojis: 518


In [136]:
df[df['has_emoji'] == True]

,Comment,Sentiment,has_emoji
47118,💩👎💩👎💩👎💩 👻☠️ impossible delete list sub lists t...,0,True
47434,reminder feature working 🥱,0,True
47480,regular user appthis application needs update ...,1,True
47576,literally blessed know todoist♥️,2,True
47587,application good past anymore stopped sync man...,0,True
...,...,...,...
237048,modi interacted citizens new delhi everyone to...,2,True
237752,might get dã©jã watching interview reminisce i...,2,True
238942,ravi listening modi â speech today must tell...,2,True
239041,à¤¿à¥ à¥à¥à¤ à¤¼à¥ à¥ à¥à¤°à¤à¤®à¤¾à¤¨...,1,True
